# 06 · Segmentation Pipeline

Full pipeline for one image stack:
split → segment → stitch → filter → measure → save CSV.

**Prerequisites:** `04_finetuning.ipynb` — or set `MODEL_PATH = None` to use base cpsam.

**Critical:** segmentation uses normalised patches; measurement uses raw uint16.
These two paths must never be swapped.

**Outputs:**
- `data/processed/<stack_name>_masks.tif` — final integer label mask
- `data/results/<stack_name>.csv` — per-cell intensities
- `figures/qc/06_filter_steps.png` — cell counts at each filter step
- `figures/qc/06_masks_overview.png` — final masks on BFP thumbnail

**Environment:** `cellpose`

In [ ]:
import numpy as np
import tifffile
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

from cellpose import models

from src.io import load_stack, split_image, stitch_masks, save_masks
from src.preprocess import normalize_for_segmentation
from src.segment import size_filter, spectral_filter, sytox_exclusion
from src.measure import extract_intensities

In [ ]:
PROJECT_ROOT = Path('/Users/alicehong/projects/cellpose-biosensor')

STACK_PATH = PROJECT_ROOT / 'data/raw/16bit/Scene-01-20260416-C3M2_Tcol_1-A01Export-01_c1-4_stack.tif'

# Fine-tuned model path — reads from notebook 04's recorded path if available
_path_file = PROJECT_ROOT / 'models/finetuned/latest_model_path.txt'
MODEL_PATH = Path(_path_file.read_text().strip()) if _path_file.exists() else None
print(f'Model: {MODEL_PATH if MODEL_PATH else "base cpsam"}')

(PROJECT_ROOT / 'data/processed').mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / 'data/results').mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / 'figures/qc').mkdir(parents=True, exist_ok=True)

PIXEL_SIZE_UM = 0.035
DIAMETER_PX   = 1.5 / PIXEL_SIZE_UM   # 42.9 px

# ── Grid split (match what was used in notebook 01) ───────────────────────────
N_ROWS = 3
N_COLS = 3

# ── Post-segmentation filter thresholds ───────────────────────────────────────
MIN_AREA      = 8      # px²  — smaller = noise
MAX_AREA      = 300    # px²  — larger = debris / host cell fragment
AUTOFL_THRESH = 0.6    # max min(GFP,RFP)/BFP per cell — tune if needed
SYTOX_OVERLAP = 0.25   # max fraction of cell overlapping SYTOX+ pixels

## Load stack and model

In [ ]:
print('Loading stack...')
stack = load_stack(STACK_PATH)
H, W  = stack.shape[1], stack.shape[2]
print(f'Stack: {stack.shape}  dtype: {stack.dtype}  max: {stack.max()}')

print('\nLoading model...')
if MODEL_PATH and MODEL_PATH.exists():
    model = models.CellposeModel(gpu=True, pretrained_model=str(MODEL_PATH))
    print(f'Loaded fine-tuned model: {MODEL_PATH.name}')
else:
    model = models.CellposeModel(gpu=True)  # base cpsam
    print('Using base cpsam (no fine-tuned model found)')

## Split and segment

BFP channel is normalised globally (p1/p99 on full image) before splitting into
N_ROWS×N_COLS patches. All patches share the same intensity scale.
Raw uint16 is preserved in `stack` for measurement.

In [ ]:
bfp_full = stack[0]                                          # raw uint16 — kept for measurement
bfp_norm = normalize_for_segmentation(bfp_full)              # global p1/p99 normalisation
patches  = split_image(bfp_norm, n_rows=N_ROWS, n_cols=N_COLS)  # float32 tiles
print(f'{N_ROWS}×{N_COLS} = {len(patches)} patches')
for p in patches:
    print(f'  r{p["row"]}c{p["col"]}  {p["h"]}×{p["w"]} px')

for i, p in enumerate(patches):
    masks_patch, _, _ = model.eval(
        p['tile'],           # already normalised float32 — do not re-normalise
        diameter=DIAMETER_PX,
        channels=[0, 0],
        normalize=False,
        flow_threshold=0.4,
        cellprob_threshold=0.0,
    )
    p['mask'] = masks_patch
    print(f'  r{p["row"]}c{p["col"]} → {masks_patch.max()} cells')

H, W = bfp_full.shape
masks_raw = stitch_masks(patches, H, W)
print(f'\nStitched: {masks_raw.shape}  total cells before filtering: {masks_raw.max()}')

## Apply post-segmentation filters

Filters applied in order (CLAUDE.md §Post-segmentation filters):
1. Size filter — remove noise and debris by area
2. Spectral filter — remove fecal autofluorescence (high min(GFP,RFP)/BFP)
3. SYTOX exclusion — remove cells overlapping host nuclei

All filters use raw uint16 channel values, not normalised.

In [ ]:
n0 = int(masks_raw.max())

print(f'Before filters:   {n0:>6} cells')

masks_sz = size_filter(masks_raw, min_area=MIN_AREA, max_area=MAX_AREA)
n1 = int(masks_sz.max())
print(f'After size filter:{n1:>6} cells  (removed {n0-n1})')

masks_sp = spectral_filter(
    masks_sz,
    bfp=stack[0], gfp=stack[1], rfp=stack[2],
    max_autofl_score=AUTOFL_THRESH)
n2 = int(masks_sp.max())
print(f'After spectral:   {n2:>6} cells  (removed {n1-n2})')

masks_final = sytox_exclusion(
    masks_sp,
    sytox=stack[3],
    max_overlap=SYTOX_OVERLAP)
n3 = int(masks_final.max())
print(f'After SYTOX excl: {n3:>6} cells  (removed {n2-n3})')
print(f'\nTotal removed: {n0-n3} / {n0}  ({100*(n0-n3)/max(n0,1):.1f}%)')

# QC bar chart
fig, ax = plt.subplots(figsize=(7, 3.5), facecolor='#0a0a0a')
ax.set_facecolor('#111')
steps = ['Raw', 'Size\nfilter', 'Spectral\nfilter', 'SYTOX\nexclusion']
counts = [n0, n1, n2, n3]
bars = ax.bar(steps, counts, color=['#888', '#4488ff', '#ff4444', '#44ff88'])
for bar, n in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(n),
            ha='center', color='white', fontsize=9)
ax.set_ylabel('Cell count', color='white')
ax.set_title('Cells retained at each filter step', color='white')
ax.tick_params(colors='#aaa')
for sp in ax.spines.values(): sp.set_edgecolor('#333')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'figures/qc/06_filter_steps.png',
            dpi=100, bbox_inches='tight', facecolor='#0a0a0a')
plt.show()

## Save masks

In [ ]:
mask_out = PROJECT_ROOT / 'data/processed' / (STACK_PATH.stem + '_masks.tif')
save_masks(masks_final, mask_out)
print(f'Masks saved → {mask_out}')

## Masks overview — spot check on downsampled image

In [ ]:
THUMB = 8  # downsample factor
thumb_bfp  = stack[0][::THUMB, ::THUMB]
thumb_norm = normalize_for_segmentation(thumb_bfp, low_pct=1, high_pct=99.9)
thumb_mask = masks_final[::THUMB, ::THUMB]

fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor='black')
for ax, mask, title in zip(axes,
                            [np.zeros_like(thumb_mask), thumb_mask],
                            ['BFP (no masks)', f'BFP + {n3} cells']):
    ax.imshow(thumb_norm, cmap='Blues', vmin=0, vmax=1)
    if mask.max() > 0:
        ax.contour(mask, levels=np.arange(0.5, mask.max() + 0.5),
                   colors='red', linewidths=0.3)
    ax.set_title(title, color='white', fontsize=10)
    ax.axis('off')

fig.suptitle(f'Final masks — {STACK_PATH.name} (1:{THUMB} thumbnail)',
             color='white', fontsize=10)
plt.tight_layout(pad=0.3)
plt.savefig(PROJECT_ROOT / 'figures/qc/06_masks_overview.png',
            dpi=120, bbox_inches='tight', facecolor='black')
plt.show()

## Extract per-cell intensities

Measurements always from raw uint16 stack — never from normalised images.

In [ ]:
print('Extracting intensities (this takes a few minutes on large masks)...')
df = extract_intensities(
    masks_final,
    stack,
    image_id=STACK_PATH.stem,
    local_bg=True,
    ring_width=5,
)

print(f'\n{len(df)} cells measured')
print(df[['area_px', 'bfp', 'gfp', 'rfp', 'gfp_over_bfp', 'rfp_over_bfp']].describe().round(2))

## Save CSV

In [ ]:
csv_out = PROJECT_ROOT / 'data/results' / (STACK_PATH.stem + '.csv')
df.to_csv(csv_out, index=False)
print(f'Saved {len(df)} rows → {csv_out}')

## Quick QC plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), facecolor='#0a0a0a')
colors = ['#4488ff', '#44ff88', '#ff4444']

# BFP distribution — should be unimodal if segmentation is clean
axes[0].set_facecolor('#111')
axes[0].hist(df['bfp'], bins=60, color='#4488ff', alpha=0.85)
axes[0].set_title('BFP (bg-subtracted)', color='white')
axes[0].set_xlabel('Intensity (ADU)', color='#aaa')
axes[0].set_ylabel('Cells', color='#aaa')
axes[0].tick_params(colors='#aaa')

# GFP/BFP ratio
axes[1].set_facecolor('#111')
valid = df['gfp_over_bfp'].dropna()
axes[1].hist(valid, bins=60, color='#44ff88', alpha=0.85)
axes[1].set_title('GFP / BFP ratio', color='white')
axes[1].set_xlabel('Ratio', color='#aaa')
axes[1].tick_params(colors='#aaa')

# RFP/BFP ratio
axes[2].set_facecolor('#111')
valid = df['rfp_over_bfp'].dropna()
axes[2].hist(valid, bins=60, color='#ff4444', alpha=0.85)
axes[2].set_title('RFP / BFP ratio', color='white')
axes[2].set_xlabel('Ratio', color='#aaa')
axes[2].tick_params(colors='#aaa')

for ax in axes:
    for sp in ax.spines.values(): sp.set_edgecolor('#333')

fig.suptitle(f'{STACK_PATH.stem} — {len(df)} cells', color='white', fontsize=10)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'figures/qc/06_intensity_qc.png',
            dpi=100, bbox_inches='tight', facecolor='#0a0a0a')
plt.show()
print(f'Saved → {PROJECT_ROOT}/figures/qc/06_intensity_qc.png')